In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'test-download-capitanata'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

# 03 - Creazione e salvataggio dataset
- estrazione dei valori spettrali Sentinel-2 per tutti i punti campionati (campionamento diretto dai GeoTIFF dell'area)
- feature engineering (serie temporali NDVI/NDWI e indici fenologici per la classificazione)

#### Caricamento ground truth e legenda

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

# definizione dei percorsi di input
ground_truth_path = DATA_DIR / "interim" / "points.json"
sentinel_area_dir = DATA_DIR / "processed" / "sentinel2_capitanata_area"
legend_path = DATA_DIR / "processed" / "legend.json"

# caricamento del file con le coordinate e i codici delle colture
ground_truth_df = pd.read_json(ground_truth_path)

# esclusione classi fittizie 'Undecided' (3100 e 3200) e 'Other Cereals' (1150)
EXCLUDED_CLASSES = [3100, 3200, 1150]
initial_count = len(ground_truth_df)
ground_truth_df = ground_truth_df[~ground_truth_df['code'].astype(int).isin(EXCLUDED_CLASSES)].reset_index(drop=True)

print(f"Campi totali in points.json: {initial_count}")
print(f"Campi scartati (classi {EXCLUDED_CLASSES}): {initial_count - len(ground_truth_df)}")
print(f"Campi validi da elaborare: {len(ground_truth_df)}")

# caricamento della legenda con i nomi delle colture
legend = {}
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)
    print(f"Legenda caricata con successo ({len(legend)} voci).")
else:
    print(f"⚠️ Legenda non trovata in {legend_path}.")

#### Estrazione dei valori spettrali dai GeoTIFF dell'area Capitanata
Campionamento ad alta velocità tramite finestra 3x3 (mediana spaziale) direttamente dai raster mensili.

In [ ]:
import rasterio
from rasterio.warp import transform
from tqdm.auto import tqdm
import re

# Mappatura indici delle bande salvate nel raster multibanda (1-based index per rasterio)
# Ordine: 1: B02 (Blu), 2: B03 (Verde), 3: B04 (Rosso), 4: B08 (NIR), 5: B11 (SWIR1), 6: B12 (SWIR2)
BAND_INDEX_MAP = {
    "B02": 1,
    "B03": 2,
    "B04": 3,
    "B08": 4,
    "B11": 5,
    "B12": 6,
}

# Struttura dati per memorizzare i valori per ogni punto e per ciascun mese (1..12)
point_monthly_values = {idx: {m: {} for m in range(1, 13)} for idx in range(len(ground_truth_df))}

# Recupera tutti i file raster del 2023 disponibili nella cartella dell'area
tif_files = sorted(list(sentinel_area_dir.glob("*.tif")))
print(f"Trovati {len(tif_files)} file GeoTIFF in: {sentinel_area_dir.resolve()}")

lons = ground_truth_df["lon"].values
lats = ground_truth_df["lat"].values

for tif_path in tqdm(tif_files, desc="Elaborazione Raster Mensili"):
    # Estrae il mese dal nome del file
    m_match = re.search(r"\d{4}[-_](\d{2})", tif_path.name)
    if not m_match:
        continue
    mo = int(m_match.group(1))

    with rasterio.open(tif_path) as src:
        xs, ys = transform("EPSG:4326", src.crs, lons, lats)
        n_bands = src.count

        for idx, (x, y) in enumerate(zip(xs, ys)):
            r_c, c_c = src.index(x, y)

            # Verifica che il punto ricada all'interno dei limiti dell'immagine
            if 0 <= r_c < src.height and 0 <= c_c < src.width:
                # Finestra 3x3 pixel centrata sulle coordinate
                r_min = max(0, r_c - 1)
                r_max = min(src.height, r_c + 2)
                c_min = max(0, c_c - 1)
                c_max = min(src.width, c_c + 2)

                window = ((r_min, r_max), (c_min, c_max))
                patch = src.read(window=window)  # shape: (n_bands, H_win, W_win)

                for b_name, b_idx in BAND_INDEX_MAP.items():
                    if b_idx <= n_bands:
                        band_vals = patch[b_idx - 1]
                        valid_vals = band_vals[band_vals > 0]
                        if len(valid_vals) > 0:
                            point_monthly_values[idx][mo][b_name] = float(np.median(valid_vals))

print("✅ Estrazione spettrale completata con successo per tutti i punti!")

#### Feature Engineering (Serie temporali NDVI/NDWI e indici fenologici)

In [ ]:
records = []

for idx, row in tqdm(ground_truth_df.iterrows(), total=len(ground_truth_df), desc="Calcolo Feature"):
    crop_id = int(row["code"])
    monthly_bands = point_monthly_values[idx]

    # --- MEDIE ANNUALI DI RIFLETTANZA PER BANDA ---
    def extract_clean_band_mean(band_name):
        vals = [
            monthly_bands[m][band_name]
            for m in range(1, 13)
            if band_name in monthly_bands[m] and monthly_bands[m][band_name] > 0
        ]
        return float(np.mean(vals)) if vals else 0.0

    record = {
        "ID_Campo": idx,
        "Ground_Truth": crop_id,
        "Crop_Name": legend.get(str(crop_id), str(crop_id)),
        "Blu_B02": extract_clean_band_mean("B02"),
        "Verde_B03": extract_clean_band_mean("B03"),
        "Rosso_B04": extract_clean_band_mean("B04"),
        "NIR_B08": extract_clean_band_mean("B08"),
        "SWIR1_B11": extract_clean_band_mean("B11"),
        "SWIR2_B12": extract_clean_band_mean("B12"),
    }

    # --- SERIE TEMPORALI MENSILI (NDVI e NDWI) ---
    raw_ndvis = {}
    raw_ndwis = {}

    for m in range(1, 13):
        red_v = monthly_bands[m].get("B04", 0.0)
        nir_v = monthly_bands[m].get("B08", 0.0)
        swir1_v = monthly_bands[m].get("B11", 0.0)

        # NDVI mensile
        if red_v > 0 and nir_v > 0:
            raw_ndvis[m] = (nir_v - red_v) / (nir_v + red_v)
        else:
            raw_ndvis[m] = np.nan

        # NDWI mensile (contenuto idrico fogliare)
        if nir_v > 0 and swir1_v > 0:
            raw_ndwis[m] = (nir_v - swir1_v) / (nir_v + swir1_v)
        else:
            raw_ndwis[m] = np.nan

    s_ndvi = pd.Series(raw_ndvis, index=range(1, 13))
    s_ndwi = pd.Series(raw_ndwis, index=range(1, 13))

    # Vengono richiesti almeno 6 mesi con osservazioni valide
    if s_ndvi.dropna().count() < 6:
        continue

    # Ricostruzione temporale continua tramite interpolazione lineare
    s_ndvi_interp = s_ndvi.interpolate(method="linear").bfill().ffill()
    s_ndwi_interp = s_ndwi.interpolate(method="linear").bfill().ffill()

    # Salvataggio delle feature mensili (01..12)
    for m in range(1, 13):
        month_str = f"{m:02d}"
        record[f"NDVI_{month_str}"] = round(float(s_ndvi_interp[m]), 4)
        record[f"NDWI_{month_str}"] = round(float(s_ndwi_interp[m]), 4)

    ndvi_arr = s_ndvi_interp.values
    ndwi_arr = s_ndwi_interp.values

    # --- INDICATORI FENOLOGICI E STRUTTURALI ---
    record["NDVI_max"] = round(float(np.max(ndvi_arr)), 4)
    record["NDVI_min"] = round(float(np.min(ndvi_arr)), 4)
    record["NDVI_amp"] = round(float(record["NDVI_max"] - record["NDVI_min"]), 4)
    record["Peak_Month"] = int(np.argmax(ndvi_arr) + 1)
    record["NDVI_mean"] = round(float(np.mean(ndvi_arr)), 4)
    record["NDVI_std"] = round(float(np.std(ndvi_arr)), 4)

    # Delta stagionali (indices: 0=Gen, 3=Apr, 4=Mag, 5=Giu, 6=Lug, 8=Set)
    record["NDVI_diff_lug_apr"] = round(float(ndvi_arr[6] - ndvi_arr[3]), 4)
    record["NDVI_diff_apr_gen"] = round(float(ndvi_arr[3] - ndvi_arr[0]), 4)
    record["NDVI_diff_set_lug"] = round(float(ndvi_arr[8] - ndvi_arr[6]), 4)
    record["Orzo_Wheat_Ratio"] = round(float(ndvi_arr[3] / (ndvi_arr[4] + 0.01)), 4)
    record["NDVI_senescence_rate"] = round(float(ndvi_arr[5] - ndvi_arr[4]), 4)

    # Indicatori di contenuto idrico (NDWI)
    record["NDWI_mean"] = round(float(np.mean(ndwi_arr)), 4)
    record["NDWI_diff_lug_gen"] = round(float(ndwi_arr[6] - ndwi_arr[0]), 4)

    # Rapporti spettrali SWIR/NIR
    nir_b8 = record["NIR_B08"]
    swir1_b11 = record["SWIR1_B11"]
    swir2_b12 = record["SWIR2_B12"]
    record["SWIR_NIR_ratio"] = round(float(swir1_b11 / (nir_b8 + 0.001) if nir_b8 > 0 else 0.0), 4)
    record["SWIR_Cellulose_ratio"] = round(float(swir2_b12 / (swir1_b11 + 0.001) if swir1_b11 > 0 else 0.0), 4)

    # Metriche specialistiche per cluster critici
    ndvi_winter = float((ndvi_arr[0] + ndvi_arr[1] + ndvi_arr[11]) / 3.0)
    ndvi_summer = float((ndvi_arr[6] + ndvi_arr[7]) / 2.0)
    record["NDVI_winter"] = round(ndvi_winter, 4)
    record["NDVI_summer_winter_diff"] = round(float(ndvi_summer - ndvi_winter), 4)
    record["NDVI_AUC"] = round(float(np.sum(ndvi_arr)), 4)
    record["Active_Months_Count"] = int(np.sum(ndvi_arr > 0.35))
    record["Senescence_May_Apr"] = round(float(ndvi_arr[4] - ndvi_arr[3]), 4)
    record["Greenup_Mar_Feb"] = round(float(ndvi_arr[2] - ndvi_arr[1]), 4)

    records.append(record)

final_df = pd.DataFrame(records)
if not final_df.empty:
    final_df = final_df.sort_values(by="ID_Campo").reset_index(drop=True)

print(f"\nElaborazione completata! Dataset creato con {len(final_df)} campi validi.")

#### Salvataggio e report del dataset finale

In [ ]:
dataset_dir = DATA_DIR / 'processed' / 'dataset'
dataset_dir.mkdir(parents=True, exist_ok=True)

parquet_path = dataset_dir / 'dataset.parquet'
csv_path = dataset_dir / 'dataset.csv'

final_df.to_parquet(parquet_path, index=False)
final_df.to_csv(csv_path, index=False)

print(f"File salvati con successo in:")
print(f"   • Parquet: {parquet_path.resolve()}")
print(f"   • CSV:     {csv_path.resolve()}")
print(f"\nDimensioni tabella finale: {final_df.shape[0]} righe x {final_df.shape[1]} colonne")

# controllo campioni estratti per ciascuna coltura
print(f"Distribuzione dei campioni estratti per classe:")
display(final_df['Crop_Name'].value_counts())

# anteprima delle prime 5 righe
display(final_df.head())